# Incremental Dataset Sampling for Supervised Learning

## Objective

In this experiment, we want to investigate how the size of the training dataset affects the performance of a supervised learning model.

Instead of training the model on the entire dataset from the beginning, we gradually increase the training data size in batches of 100,000 samples.

The experiments will use the following training sizes:

- 100,000 samples
- 200,000 samples
- 300,000 samples
- 400,000 samples
- And so on, until the desired performance is achieved.

The goal is to identify a suitable training dataset size without unnecessarily using the entire dataset.

---

## 1. Creating a Fixed Validation Set

First, we create a separate validation set containing 15,000 labeled samples.

The validation set is sampled with approximately balanced classes to ensure that both recommendation categories are sufficiently represented during model evaluation.

**Important points:**

- The validation set is created only once.
- Its samples are excluded from all training batches.
- The same validation set is used in every experiment.
- The validation set is not used for model training.

Keeping the validation set fixed allows us to compare model performance across different training dataset sizes more reliably.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# File paths
SOURCE_PATH = Path("../data/digikala-comments-final.csv")
DATA_DIR = Path("../data")

# Dataset settings
ID_COL = "id"
TARGET_COL = "recommendation_status"
RATING_COL = "rate"
TEXT_COL = "text"
RANDOM_SEED = 42


In [69]:
def split_by_product_id(
    source_path,
    sample_size,
    excluded_product_ids=None,
    seed=42,
    labels=None,
    chunk_size=100_000
):
    """
    Create a balanced sample while keeping product groups separate.
    """

    if excluded_product_ids is None:
        excluded_product_ids = set()

    if labels is None:
        labels = [
            "recommended",
            "not_recommended",
            "no_idea"
        ]

    # Calculate the quota for each class
    base_quota = sample_size // len(labels)
    remainder = sample_size % len(labels)

    quotas = {
        label: base_quota + (i < remainder)
        for i, label in enumerate(labels)
    }

    # Detect the text column
    source_columns = pd.read_csv(
        source_path,
        nrows=0
    ).columns.tolist()

    text_source_col = (
        "text" if "text" in source_columns else "body_normalized"
    )

    required_columns = [
        ID_COL,
        "product_id",
        text_source_col,
        TARGET_COL,
        RATING_COL
    ]

    missing_columns = [
        col for col in required_columns
        if col not in source_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    # --------------------------------------------------
    # Pass 1: Count comments per product and class
    # --------------------------------------------------

    product_count_parts = []
    seen_ids = set()

    for chunk in pd.read_csv(
        source_path,
        usecols=required_columns,
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False
    ):

        chunk[ID_COL] = chunk[ID_COL].str.strip()
        chunk["product_id"] = chunk["product_id"].str.strip()

        chunk[TARGET_COL] = (
            chunk[TARGET_COL].str.strip().str.lower()
        )

        # Keep valid comments and available products
        chunk = chunk[
            chunk[ID_COL].ne("")
            & chunk["product_id"].ne("")
            & chunk[text_source_col].str.strip().ne("")
            & chunk[TARGET_COL].isin(labels)
            & ~chunk["product_id"].isin(excluded_product_ids)
        ].copy()

        # Remove duplicate comment IDs
        chunk = chunk[
            ~chunk[ID_COL].isin(seen_ids)
            & ~chunk[ID_COL].duplicated()
        ]

        seen_ids.update(chunk[ID_COL])

        if chunk.empty:
            continue

        # Count comments of each class for every product
        counts = (
            chunk.groupby(["product_id", TARGET_COL])
            .size()
            .unstack(fill_value=0)
        )

        product_count_parts.append(counts)

    if not product_count_parts:
        raise ValueError("No available comments found.")

    product_stats = (
        pd.concat(product_count_parts)
        .groupby(level=0)
        .sum()
        .reindex(columns=labels, fill_value=0)
    )

    # Check whether enough comments are available
    available_counts = product_stats.sum()

    print("Available comments:")
    print(available_counts)

    for label in labels:
        if available_counts[label] < quotas[label]:
            raise ValueError(
                f"Not enough comments for {label}. "
                f"Available: {available_counts[label]}, "
                f"Required: {quotas[label]}"
            )

    # --------------------------------------------------
    # Pass 2: Select product groups
    # --------------------------------------------------

    quota_series = pd.Series(quotas)

    product_scores = (
    product_stats.clip(upper=quota_series, axis=1)
    .div(quota_series, axis=1)
    .sum(axis=1)
    )
    rng = np.random.default_rng(seed)

    ranked_products = product_stats.copy()

    ranked_products["_score"] = product_scores
    ranked_products["_random"] = rng.random(
        len(ranked_products)
    )

    ranked_products = ranked_products.sort_values(
        ["_score", "_random"],
        ascending=False
    )

    selected_product_ids = set()

    selected_counts = {
        label: 0 for label in labels
    }

    for product_id, row in ranked_products.iterrows():

        # Stop when all class quotas are satisfied
        if all(
            selected_counts[label] >= quotas[label]
            for label in labels
        ):
            break

        # Check whether this product contributes to a needed class
        contributes = any(
            row[label] > 0
            and selected_counts[label] < quotas[label]
            for label in labels
        )

        if not contributes:
            continue

        selected_product_ids.add(product_id)

        for label in labels:
            remaining = quotas[label] - selected_counts[label]

            selected_counts[label] += min(
                int(row[label]),
                remaining
            )

    for label in labels:
        if selected_counts[label] < quotas[label]:
            raise RuntimeError(
                f"Could not select enough products for {label}."
            )

    # --------------------------------------------------
    # Pass 3: Collect comments from selected products
    # --------------------------------------------------

    selected_parts = []
    seen_sample_ids = set()

    for chunk in pd.read_csv(
        source_path,
        usecols=required_columns,
        chunksize=chunk_size,
        dtype=str,
        keep_default_na=False
    ):

        chunk[ID_COL] = chunk[ID_COL].str.strip()
        chunk["product_id"] = chunk["product_id"].str.strip()

        chunk[TARGET_COL] = (
            chunk[TARGET_COL].str.strip().str.lower()
        )

        chunk = chunk[
            chunk["product_id"].isin(selected_product_ids)
            & chunk[TARGET_COL].isin(labels)
            & chunk[ID_COL].ne("")
            & chunk[text_source_col].str.strip().ne("")
        ].copy()

        chunk = chunk[
            ~chunk[ID_COL].isin(seen_sample_ids)
            & ~chunk[ID_COL].duplicated()
        ]

        seen_sample_ids.update(chunk[ID_COL])

        if not chunk.empty:
            selected_parts.append(chunk)

    if not selected_parts:
        raise RuntimeError("No comments found for selected products.")

    candidate_df = pd.concat(
        selected_parts,
        ignore_index=True
    )

    # --------------------------------------------------
    # Pass 4: Select the exact quota for each class
    # --------------------------------------------------

    sample_parts = []

    for i, label in enumerate(labels):

        class_df = candidate_df[
            candidate_df[TARGET_COL] == label
        ]

        if len(class_df) < quotas[label]:
            raise RuntimeError(
                f"Not enough candidates for {label}."
            )

        class_sample = class_df.sample(
            n=quotas[label],
            random_state=seed + i
        )

        sample_parts.append(class_sample)

    sample_df = pd.concat(
        sample_parts,
        ignore_index=True
    )

    sample_df = sample_df.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    # Keep required columns
    sample_df = sample_df[
        [
            ID_COL,
            "product_id",
            text_source_col,
            TARGET_COL,
            RATING_COL
        ]
    ].rename(
        columns={text_source_col: TEXT_COL}
    )

    # Convert rating values to numeric
    sample_df[RATING_COL] = pd.to_numeric(
        sample_df[RATING_COL],
        errors="coerce"
    )

    return sample_df, selected_product_ids

In [70]:
# Define validation settings
VALIDATION_SIZE = 50_000

LABELS = [
    "recommended",
    "not_recommended",
    "no_idea"
]

# No products are reserved at the beginning
reserved_product_ids = set()

# Create validation dataset
validation_df, validation_product_ids = split_by_product_id(
    source_path=SOURCE_PATH,
    sample_size=VALIDATION_SIZE,
    excluded_product_ids=reserved_product_ids,
    seed=RANDOM_SEED,
    labels=LABELS,
    chunk_size=VALIDATION_SIZE
)

# Reserve selected validation products
reserved_product_ids.update(validation_product_ids)

# Save validation dataset
VALIDATION_PATH = DATA_DIR / "validation_50k.csv"

validation_df.to_csv(
    VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save reserved product IDs
RESERVED_PRODUCTS_PATH = DATA_DIR / "reserved_product_ids.csv"

pd.DataFrame({
    "product_id": sorted(reserved_product_ids)
}).to_csv(
    RESERVED_PRODUCTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Display results
print("Validation shape:", validation_df.shape)

print("\nClass distribution:")
print(
    validation_df[TARGET_COL]
    .value_counts()
    .reindex(LABELS, fill_value=0)
)

print("\nUnique products:", validation_df["product_id"].nunique())
print("Reserved products:", len(reserved_product_ids))

print("\nSaved:", VALIDATION_PATH.name)

Available comments:
recommendation_status
recommended        3200264
not_recommended     452496
no_idea             467660
dtype: int64
Validation shape: (50000, 5)

Class distribution:
recommendation_status
recommended        16667
not_recommended    16667
no_idea            16666
Name: count, dtype: int64

Unique products: 999
Reserved products: 999

Saved: validation_50k.csv


In [ ]:
df = pd.read_csv()

## 2. Creating Training Batches

After creating the validation set, we generate training batches.

Each batch contains 100,000 labeled samples and is sampled with approximately balanced classes.

To prevent data leakage and repeated samples:

- Validation IDs are excluded from training.
- IDs from previously generated training batches are excluded.
- Each training sample is assigned a unique ID.
- The selected IDs are stored in `selected_train_ids.csv`.

The original dataset remains unchanged. Instead of deleting selected rows from the source file, we use their IDs to keep track of previously selected samples.

### Reproducibility

A fixed random seed is used during sampling.

This helps make the sampling process reproducible, provided that the source file, its row order, chunk size, and random seed remain unchanged.

In [134]:
# Create a training batch

CHUNK_SIZE = 100_000
BATCH_NUMBER = 10

BATCH_PATH = DATA_DIR / (
    f"train_batch_{BATCH_NUMBER:02d}.csv"
)

if BATCH_PATH.exists():
    raise FileExistsError(
        f"{BATCH_PATH.name} already exists. "
        "Choose a different batch number."
    )

# --------------------------------------------------
# Load previously reserved product IDs
# --------------------------------------------------

if RESERVED_PRODUCTS_PATH.exists():

    reserved_product_ids = set(
        pd.read_csv(
            RESERVED_PRODUCTS_PATH,
            dtype=str
        )["product_id"].str.strip()
    )

else:
    raise FileNotFoundError(
        "Reserved product IDs file not found. "
        "Create Validation first."
    )

print("Previously reserved products:", len(reserved_product_ids))

# --------------------------------------------------
# Create a new training batch
# --------------------------------------------------

batch_df, batch_product_ids = split_by_product_id(
    source_path=SOURCE_PATH,
    sample_size=BATCH_SIZE,
    excluded_product_ids=reserved_product_ids,
    seed=RANDOM_SEED + BATCH_NUMBER,
    labels=LABELS,
    chunk_size=CHUNK_SIZE
)

# --------------------------------------------------
# Verify the new batch
# --------------------------------------------------

assert len(batch_df) == BATCH_SIZE
assert batch_df[ID_COL].is_unique

# Verify product separation
assert batch_product_ids.isdisjoint(
    reserved_product_ids
)

# Get actual product IDs from the final batch
actual_product_ids = set(
    batch_df["product_id"].astype(str).str.strip()
)

# Normalize selected product IDs
selected_product_ids = set(
    str(pid).strip() for pid in batch_product_ids
)

# Compare the two sets
missing_from_batch = selected_product_ids - actual_product_ids
unexpected_in_batch = actual_product_ids - selected_product_ids

print("Selected product IDs:", len(selected_product_ids))
print("Actual batch product IDs:", len(actual_product_ids))

print("Selected products not present in batch:", len(missing_from_batch))
print("Unexpected products in batch:", len(unexpected_in_batch))

# Verify that every product in the batch was selected
assert actual_product_ids.issubset(selected_product_ids)

# --------------------------------------------------
# Save the new batch
# --------------------------------------------------

batch_df.to_csv(
    BATCH_PATH,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# Update reserved product IDs
# --------------------------------------------------

reserved_product_ids.update(batch_product_ids)

pd.DataFrame({
    "product_id": sorted(reserved_product_ids)
}).to_csv(
    RESERVED_PRODUCTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# Show results
# --------------------------------------------------

print(f"\nBatch {BATCH_NUMBER} created successfully!")

print("\nBatch shape:", batch_df.shape)

print("\nClass distribution:")
print(
    batch_df[TARGET_COL]
    .value_counts()
    .reindex(LABELS, fill_value=0)
)

print("\nUnique comment IDs:", batch_df[ID_COL].nunique())

print("Unique product IDs:", batch_df["product_id"].nunique())

print(
    "\nNewly reserved products:",
    len(batch_product_ids)
)

print(
    "Total reserved products:",
    len(reserved_product_ids)
)

print("\nSaved file:", BATCH_PATH.name)

Previously reserved products: 44866
Available comments:
recommendation_status
recommended        779170
not_recommended    135770
no_idea            126302
dtype: int64
Selected product IDs: 16600
Actual batch product IDs: 16591
Selected products not present in batch: 9
Unexpected products in batch: 0

Batch 10 created successfully!

Batch shape: (100000, 5)

Class distribution:
recommendation_status
recommended        33334
not_recommended    33333
no_idea            33333
Name: count, dtype: int64

Unique comment IDs: 100000
Unique product IDs: 16591

Newly reserved products: 16600
Total reserved products: 61466

Saved file: train_batch_10.csv


## 3. Building Training Sets for Experiments

Once the training batches have been generated, we can combine them to create training datasets of different sizes.

For example:

| Experiment | Training batches | Training size |
|---|---:|---:|
| 1 | 1 | 100,000 |
| 2 | 2 | 200,000 |
| 3 | 3 | 300,000 |
| 4 | 4 | 400,000 |

For each experiment, all batches from the beginning up to the selected batch number are combined.

**Important:** The model must be trained from scratch for each experiment. We should not train the previous model only on the newly added batch, because that would make the experiments less comparable.



In [135]:
from pathlib import Path
import pandas as pd

# Choose the current experiment
BATCH_COUNT = 10

# Create paths for the selected training batches
batch_paths = [
    DATA_DIR / f"train_batch_{i:02d}.csv"
    for i in range(1, BATCH_COUNT + 1)
]

# Check whether all batch files exist
for path in batch_paths:
    if not path.exists():
        raise FileNotFoundError(f"Batch file not found: {path}")

# Load and combine all selected batches
train_df = pd.concat(
    [
        pd.read_csv(path, dtype=str, keep_default_na=False)
        for path in batch_paths
    ],
    ignore_index=True
)

# Check for duplicate comment IDs
assert train_df[ID_COL].is_unique, "Duplicate comment IDs found!"

# Load Validation product IDs
validation_df = pd.read_csv(
    VALIDATION_PATH,
    usecols=["product_id"],
    dtype=str,
    keep_default_na=False
)

validation_product_ids = set(
    validation_df["product_id"].str.strip()
)

# Check for product overlap between Train and Validation
train_product_ids = set(
    train_df["product_id"].str.strip()
)

overlap = train_product_ids.intersection(validation_product_ids)

assert not overlap, (
    f"Product leakage detected: "
    f"{len(overlap)} products appear in both Train and Validation."
)

print("No product overlap between Train and Validation.")

# Save the training dataset for this experiment
train_path = DATA_DIR / f"train_experiment_{BATCH_COUNT * 100}k.csv"

train_df.to_csv(
    train_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nTraining dataset:", train_path.name)
print("Shape:", train_df.shape)

print("\nClass distribution:")
print(train_df[TARGET_COL].value_counts())

No product overlap between Train and Validation.

Training dataset: train_experiment_1000k.csv
Shape: (1000000, 5)

Class distribution:
recommendation_status
recommended        333340
no_idea            333330
not_recommended    333330
Name: count, dtype: int64


## 4. Experimental Evaluation

For each training dataset size:

1. Train a new model using the corresponding training set.
2. Evaluate the model on the same validation set.
3. Record the evaluation metrics.
4. Compare the results with previous experiments.

Useful evaluation metrics include:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- ROC-AUC
- PR-AUC

Since the training samples are approximately balanced, but the real-world dataset may have a different class distribution, evaluation should also be performed on data that reflects the natural class distribution.

The validation set should remain unchanged throughout these experiments.

---


### Modeling 

#### Imports

In [11]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT / "src"))

from tfidf_baseline import (
    load_data,
    create_rating_band,
    prepare_data,
    train_all_models
)

In [136]:
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

# Create results directory if it does not exist
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Define dataset paths
TRAIN_PATH = DATA_DIR / "train_experiment_1000k.csv"
VALIDATION_PATH = DATA_DIR / "validation_50k_corrected.csv"

# Load training and validation datasets
train_df, val_df = load_data(
    TRAIN_PATH,
    VALIDATION_PATH
)


Training shape: (1000000, 5)
Validation shape: (50000, 5)


In [137]:
# Create rating bands
train_df = create_rating_band(train_df)
val_df = create_rating_band(val_df)

# Prepare data for TF-IDF models
train_df = prepare_data(train_df)
val_df = prepare_data(val_df)

# Display dataset information
print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTraining class distribution:")
print(train_df["recommendation_status"].value_counts())

print("\nValidation class distribution:")
print(val_df["recommendation_status"].value_counts())

Training shape: (1000000, 6)
Validation shape: (50000, 6)

Training class distribution:
recommendation_status
recommended        333340
no_idea            333330
not_recommended    333330
Name: count, dtype: int64

Validation class distribution:
recommendation_status
not_recommended    17391
recommended        17123
no_idea            15486
Name: count, dtype: int64


In [139]:
results = train_all_models(
    train_df=train_df,
    val_df=val_df
)


Training model for: recommendation_status

Target: recommendation_status
Training samples: 1,000,000
Validation samples: 50,000

Classification Report:
                 precision    recall  f1-score   support

        no_idea       0.63      0.68      0.65     15486
not_recommended       0.80      0.78      0.79     17391
    recommended       0.85      0.82      0.84     17123

       accuracy                           0.76     50000
      macro avg       0.76      0.76      0.76     50000
   weighted avg       0.77      0.76      0.76     50000

Confusion Matrix:
[[10454  2877  2155]
 [ 3544 13480   367]
 [ 2606   411 14106]]

Training model for: rating_band

Target: rating_band
Training samples: 1,000,000
Validation samples: 50,000

Classification Report:
              precision    recall  f1-score   support

        high       0.63      0.74      0.68     14506
         low       0.73      0.63      0.68     19509
      medium       0.54      0.54      0.54     15985

    accuracy

In [140]:
# Convert model results into a DataFrame
results_df = pd.DataFrame([
    {
        "target": target,
        **result["metrics"]
    }
    for target, result in results.items()
])
results_df


,target,actual_train_size,validation_size,accuracy,macro_f1,balanced_accuracy
0,recommendation_status,1000000,50000,0.76080,0.758893,0.757993
1,rating_band,1000000,50000,0.63404,0.632354,0.637683


In [141]:
output_path = RESULTS_DIR / "tfidf_baseline_results.csv"

results_df.to_csv(
    output_path,
    mode="a",
    header=not output_path.exists(),
    index=False
)

print("New results appended to:", output_path)

New results appended to: d:\AI Bootcamp_M Sharif\digikala_reviews_project\results\tfidf_baseline_results.csv


In [142]:
import pandas as pd
result = pd.read_csv("D:\\AI Bootcamp_M Sharif\\digikala_reviews_project\\results\\tfidf_baseline_results.csv")
display(result)

,target,actual_train_size,validation_size,accuracy,macro_f1,balanced_accuracy
0,recommendation_status,100000,50000,0.74614,0.744313,0.743309
1,rating_band,100000,50000,0.62156,0.619386,0.623279
2,recommendation_status,200000,50000,0.75212,0.750218,0.749240
3,rating_band,200000,50000,0.62422,0.621855,0.626646
4,recommendation_status,300000,50000,0.75420,0.752179,0.751263
5,rating_band,300000,50000,0.62602,0.623584,0.628751
6,recommendation_status,400000,50000,0.75512,0.753049,0.752156
7,rating_band,400000,50000,0.62848,0.626185,0.631409
8,recommendation_status,500000,50000,0.75682,0.754782,0.753892
9,rating_band,500000,50000,0.62968,0.627524,0.632393


### Conclusion: Impact of Training Dataset Size

To investigate the effect of training dataset size on classification performance, TF-IDF models with Logistic Regression were trained using different training sizes, ranging from 100,000 to 1,000,000 samples. The validation dataset remained fixed at 50,000 samples throughout all experiments.

The results demonstrate a consistent improvement in both classification tasks as the training dataset size increases.

For **Recommendation Status Classification**, Macro F1 increased from 0.7443 with 100K training samples to 0.7589 with 1M samples. Similarly, for **Rating Band Classification**, Macro F1 improved from 0.6194 to 0.6324.

Although the 1M training dataset achieved the highest performance, doubling the training data from 500K to 1M resulted in relatively small improvements in both tasks. This suggests diminishing returns from increasing the training dataset size.

Based on these results, **500K samples were selected as the primary training dataset for subsequent experiments**. This size provides a reasonable balance between classification performance and computational efficiency, particularly considering the higher computational cost of Transformer fine-tuning.

The 1M dataset will remain available for additional experiments to investigate whether larger training datasets provide further improvements in Transformer-based classification.

Overall, these experiments establish a TF-IDF baseline and provide a reference point for evaluating the effectiveness of the Transformer model.
